# EgoObjects eval: egocentric boxes, federated labels, classes matched to PROCTHOR30

[EgoObjects](https://github.com/facebookresearch/EgoObjects) v1.0's eval split: frames of wearable-camera videos,
mostly indoors, 368 classes, boxes only, in LVIS's federated format (`neg_category_ids`, `not_exhaustive_category_ids`
per image). Read with `habitat_embodied_sl/static_datasets/egoobjects.py`'s `egoobjects_source`, its category ids
turned into class indices, on `$HABITAT_DATA/egoobjects/data/EgoObjectsV1_unified_eval.json` and its images in
`$HABITAT_DATA/egoobjects/images/`. Needs `INSTALL.MD`'s EgoObjects section (behind Meta's license form) run first.

In [ ]:
import os
import sys
from pathlib import Path

REPO_ROOT = Path.cwd().parent
sys.path.insert(0, str(REPO_ROOT))
os.environ.setdefault("BASE_DIR", str(REPO_ROOT))

from common.env_utils.vocab_constants import CLASS_LABELS_PROCTHOR30
from common.vision.detic import matched_classes
from habitat_embodied_sl.static_datasets.detection_sources import showing_records
from habitat_embodied_sl.static_datasets.inspection import matched_table, plot_matched_counts, plot_verification, show_records

VOCAB, THRESHOLD = CLASS_LABELS_PROCTHOR30, 0.9

In [ ]:
from habitat_embodied_sl.static_datasets.egoobjects import EGOOBJECTS_ROOT, egoobjects_source

source = egoobjects_source(EGOOBJECTS_ROOT / "data" / "EgoObjectsV1_unified_eval.json", EGOOBJECTS_ROOT / "images")
records, egoobjects = source.records, source.vocabulary
print(f"{len(records)} images, {sum(len(r['annotations']) for r in records)} boxes, {len(egoobjects.classes)} classes")
{key: value for key, value in records[0].items() if key != "annotations"}

## A few examples

Random records, drawn with detectron2's `Visualizer` (masks where the source has them).

In [ ]:
show_records(records, egoobjects.classes)

## Classes matched to PROCTHOR30

`common/vision/detic.py`'s `matched_classes`: every EgoObjects class whose CLIP text embedding is at least `THRESHOLD`
cosine-similar to some `VOCAB` word's, both prompted as `a <name>` -- the classes `egoobjects.py` starts from with
`--vocab` and `--sim-word-thresh`. Change `VOCAB`/`THRESHOLD` above to try another vocabulary.

In [ ]:
matches = matched_classes(egoobjects, VOCAB, THRESHOLD)
matched = {index for index, _, _ in matches}
matched_table(egoobjects, matches)

## Matched classes' frequency

Object count of each matched class over the images showing at least one (`detection_sources.py`'s `showing_records`,
every one here, as without `--max-images`). `egoobjects.py` keeps the classes with at least `MIN_OCCURRENCES` objects
(`vocabulary.py`, after merging spelling variants), in blue; the grey ones' objects are left out of its test set.

In [ ]:
showing = showing_records(records, matched, None)
plot_matched_counts(showing, egoobjects.classes, matched)

## Federated verification

EgoObjects only verifies a class on an image where it is annotated exhaustively or listed absent (`neg_category_ids`). On
every other image -- annotated but in `not_exhaustive_category_ids`, or neither annotated nor listed -- `egoobjects.py` gives
the class's category a whole-image `iscrowd` box, so COCOeval ignores its unmatched detections there, as LVIS's own
evaluation does: only a detection on an image where the class is listed absent counts as a false positive. A class
unverified on most images is scored almost only on its recall.

In [ ]:
plot_verification(showing, egoobjects.classes, matched)

## Building the test set

```bash
PYTHONPATH=. python habitat_embodied_sl/static_datasets/egoobjects.py datasets/egoobjects_eval --vocab <VOCAB words> --sim-word-thresh 0.9
```
writes `datasets/egoobjects_eval/test/test.json` and its images in `datasets/egoobjects_eval/images/`, registered as `egoobjects_eval_test`
(`common/env_utils/dataset_registry.py`) for a training config's `DATASETS.TEST`.